# Downloading data

In [3]:
import polars as pl

path = "data/canonical_items.parquet"
df_lazy = pl.scan_parquet(path)
df_head = df_lazy.head(5).collect()

# Use a context manager to temporarily show all columns (tbl_cols=-1) and rows (tbl_rows=-1)
with pl.Config(tbl_cols=-1, tbl_rows=-1):
  print(df_head)

shape: (5, 16)
┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬────────┐
│ ite ┆ 100 ┆ 101 ┆ 102 ┆ 112 ┆ 114 ┆ 115 ┆ 116 ┆ 117 ┆ 118 ┆ 119 ┆ 120 ┆ 121 ┆ 122 ┆ mm_ ┆ has_mm │
│ m_i ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ --- ┆ emb ┆ _emb   │
│ d   ┆ i64 ┆ i64 ┆ i64 ┆ i64 ┆ i64 ┆ i64 ┆ i64 ┆ i64 ┆ i64 ┆ i64 ┆ i64 ┆ i64 ┆ i64 ┆ --- ┆ ---    │
│ --- ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆ arr ┆ bool   │
│ i64 ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆ ay[ ┆        │
│     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆ f32 ┆        │
│     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆ ,   ┆        │
│     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆ 102 ┆        │
│     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆     ┆   

In [2]:
from pathlib import Path
from datasets import get_dataset_config_names, load_dataset


import h5py
import numpy as np
import pyarrow.parquet as pq
import pyarrow as pa

In [ ]:
dataset_id: str = "TAAC2025/TencentGR-1M"
output_dir: Path = Path("./tencent_gr_parquet")
output_dir.mkdir(parents=True, exist_ok=True)

# 1. Get all available subset/configuration names
subsets: list[str] = get_dataset_config_names(dataset_id)
print(f"Found subsets: {subsets}")

# 2. Loop through each subset
for subset in subsets:
  print(f"\nProcessing subset: {subset}...")
  try:
    # Load all splits for the current subset
    dataset_dict = load_dataset(dataset_id, subset)

    # 3. Explicitly convert split_name to str to satisfy the type checker
    for split_name, dataset in dataset_dict.items():
      split_dir = output_dir / str(split_name)
      split_dir.mkdir(parents=True, exist_ok=True)

      # Save file inside the split directory
      file_path = split_dir / f"{subset}.parquet"
      dataset.to_parquet(str(file_path))
      print(f"Saved -> {file_path}")

  except Exception as e:
    print(f"Error processing subset {subset}: {e}")

print("\nAll subsets successfully downloaded and organized by split directories!")

In [5]:
import h5py
import numpy as np
import pyarrow.parquet as pq

def convert_mm_emb_to_hdf5_native(parquet_path: str, h5_output_path: str, expected_dim: int = 1024, chunk_size: int = 4096):
    """
    Ingests mm_emb parquet files using native PyArrow column iteration 
    to avoid Pandas memory realloc/fragmentation errors.
    """
    print(f"Opening mm_emb parquet file: {parquet_path}")
    parquet_file = pq.ParquetFile(parquet_path)
    total_rows = parquet_file.metadata.num_rows
    print(f"Total items in mm_emb: {total_rows}")
    
    with h5py.File(h5_output_path, 'w') as f:
        dt_str = h5py.string_dtype(encoding='utf-8')
        oid_ds = f.create_dataset(
            'anonymous_cid', 
            shape=(total_rows,), 
            dtype=dt_str, 
            chunks=(chunk_size,), 
            compression='lzf'
        )
        
        embeddings_ds = f.create_dataset(
            'embedding', 
            shape=(total_rows, expected_dim), 
            dtype=np.float32, 
            chunks=(chunk_size, expected_dim), 
            compression='lzf'
        )
        
        written_rows = 0
        skipped_rows = 0
        
        # Iterate through row groups instead of raw batches to prevent memory spikes
        for rg_idx in range(parquet_file.num_row_groups):
            row_group = parquet_file.read_row_group(rg_idx, columns=['anonymous_cid', 'emb'])
            
            oids_chunk = row_group.column('anonymous_cid').to_numpy()
            emb_chunk = row_group.column('emb')
            
            # Process sub-chunks to manage memory cleanly
            num_rows_in_rg = row_group.num_rows
            for start in range(0, num_rows_in_rg, chunk_size):
                end = min(start + chunk_size, num_rows_in_rg)
                
                sub_oids = oids_chunk[start:end]
                sub_emb_list = emb_chunk.slice(start, end)
                
                clean_oids = []
                clean_embs = []
                
                for oid, emb_val in zip(sub_oids, sub_emb_list):
                    if emb_val is None:
                        skipped_rows += 1
                        continue
                    
                    # Convert pyarrow scalar list to numpy array
                    emb_arr = np.array(emb_val.as_py(), dtype=np.float32)
                    
                    if emb_arr.ndim != 1 or emb_arr.shape[0] != expected_dim:
                        skipped_rows += 1
                        continue
                        
                    clean_oids.append(str(oid))
                    clean_embs.append(emb_arr)
                
                batch_len = len(clean_oids)
                if batch_len > 0:
                    end_pos = written_rows + batch_len
                    oid_ds[written_rows:end_pos] = np.array(clean_oids, dtype=str)
                    embeddings_ds[written_rows:end_pos] = np.stack(clean_embs)
                    written_rows += batch_len
                    
            print(f"Processed row group {rg_idx+1}/{parquet_file.num_row_groups} | Valid written: {written_rows} | Skipped: {skipped_rows}", end='\r')
            
    # Resize datasets if any rows were skipped
    if skipped_rows > 0:
        print(f"\n[Notice] Total skipped rows: {skipped_rows}. Resizing HDF5 datasets...")
        with h5py.File(h5_output_path, 'a') as f:
            f['anonymous_cid'].resize((written_rows,))
            f['embedding'].resize((written_rows, expected_dim))
            
    print(f"\nSuccessfully built HDF5 store at: {h5_output_path} with {written_rows} valid records.")

In [ ]:
convert_mm_emb_to_hdf5_native(
    parquet_path='tencent_gr_parquet/train/mm_emb_82_1024.parquet',
    h5_output_path='tencent_gr_parquet/train/mm_emb_82.h5',
    expected_dim=1024,
    chunk_size=4096
)

In [ ]:
import h5py
import numpy as np

h5_path = 'tencent_gr_parquet/train/mm_emb_82.h5'

with h5py.File(h5_path, 'r') as f:
    print("--- HDF5 Store Structure ---")
    print("Keys in HDF5 file:", list(f.keys()))
    
    oid_ds = f['anonymous_cid']
    emb_ds = f['embedding']
    
    print(f"Dataset 'anonymous_cid' shape: {oid_ds.shape} | dtype: {oid_ds.dtype}")
    print(f"Dataset 'embedding'     shape: {emb_ds.shape}     | dtype: {emb_ds.dtype}")
    print(f"Configured Chunk Sizes -> OID: {oid_ds.chunks} | Embedding: {emb_ds.chunks}")
    
    # 1. Spot-check the first few items
    sample_oids = oid_ds[:5]
    sample_embs = emb_ds[:5]
    print("\n--- Sample Data Verification ---")
    print("First 5 OIDs (anonymous_cid):", sample_oids)
    print("Embeddings slice shape for first 5:", sample_embs.shape)
    print("First 5 values of embedding[0]:", sample_embs[0, :5])
    
    # 2. Numerical health check (checking a mid-range slice for NaNs or Infs)
    mid_slice = emb_ds[2000000:2005000]
    print("\n--- Numerical Health Check (5,000 items slice) ---")
    print("Contains NaNs? :", np.isnan(mid_slice).any())
    print("Contains Infs? :", np.isinf(mid_slice).any())
    print(f"Value Range    : [{mid_slice.min():.4f}, {mid_slice.max():.4f}]")
    print(f"Mean Embedding : {mid_slice.mean():.4f}")

print("\nValidation completed successfully! The HDF5 store is fully optimized for PyTorch streaming.")